# Phân tích khám phá dữ liệu khoa học — EDA tiếng Việt

Notebook này hiển thị kết quả phân tích khám phá dữ liệu theo 17 phần, kèm bảng bằng chứng, biểu đồ dựng từ registry và thẻ chuyện khoa học 9 bước. Toàn bộ số liệu, hình ảnh và phát hiện đều được tính trong các module Python dưới `src/eda/script`; các ô mã ở đây chỉ gọi facade hiển thị.

Quy ước bất biến: nguồn dữ liệu thô chỉ được đọc; split Public Test không có đáp án tham chiếu và không được truy cập, suy diễn hay dựng nhãn; mọi biểu đồ không áp dụng đều ghi rõ lý do.


In [ ]:
import sys
from pathlib import Path

SCRIPT_ROOT = Path.cwd() / 'src' / 'eda' / 'script'
if not SCRIPT_ROOT.exists():
    SCRIPT_ROOT = Path.cwd().resolve()
    for candidate in SCRIPT_ROOT.parents:
        if (candidate / 'src' / 'eda' / 'script').is_dir():
            SCRIPT_ROOT = candidate / 'src' / 'eda' / 'script'
            break
if str(SCRIPT_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPT_ROOT))

from IPython.display import HTML, Markdown, display

from common.config import load_config
from orchestration.pipeline import run_eda

config = load_config()
run = run_eda(config)
run.run_id


In [ ]:
def _show_figure(chart_id, split='public_test', limit=1):
    html = run.figures_html(chart_id, split, limit)
    if html:
        display(HTML(html))


In [ ]:
def _show_table(table_id, split=None):
    html = run.table_html(table_id, split)
    if not html:
        print(f'Bảng {table_id} không có dữ liệu cho split {split}')
        return
    display(HTML(html))


## 00. Cấu hình, chạy pipeline và provenance

**Mục tiêu:** Xác lập cấu hình, chạy pipeline EDA một lần và hiển thị ledger kiểm định cùng provenance của toàn bộ artifact sinh ra.

**Phương pháp:** Pipeline đọc nguồn ở chế độ chỉ đọc, tính toàn bộ chỉ số tại thời điểm nạp dữ liệu và ghi registry có hash cho từng artifact.

**Cách đọc và hạn chế:** Mọi con số trong notebook đều truy vết được về run_id, source hash và phương pháp tính; không có số liệu nào nhập tay.


In [ ]:
_show_table('TBL20')


## 01. Tổng quan bộ dữ liệu

**Mục tiêu:** Mô tả phạm vi dữ liệu theo split, đơn vị đếm và mức độ bao phủ của từng doanh nghiệp.

**Phương pháp:** Đếm đối tượng siêu dữ liệu từ manifest và registry của từng split; số tệp và số giao dịch là hai đại lượng khác nhau.

**Cách đọc và hạn chế:** Tổng số tệp, số worksheet và số đối tượng CSDL là số đối tượng nguồn, không phải số giao dịch và không được cộng chéo.


In [ ]:
_show_table('TBL01')

_show_figure('INV01', 'public_test', 1)
_show_figure('GLOBAL06', 'public_test', 1)


## 02. Kiểm kê nguồn dữ liệu

**Mục tiêu:** Liệt kê toàn bộ tệp, worksheet và đối tượng CSDL kèm trạng thái đọc, hash và siêu dữ liệu cấu trúc.

**Phương pháp:** Registry kiểm kê gom từ manifest nguồn và registry worksheet/database; mỗi bản ghi gắn content hash của tệp gốc.

**Cách đọc và hạn chế:** Nhiều tệp hơn không đồng nghĩa nhiều giao dịch hơn; tệp trong manifest được đọc ở chế độ chỉ đọc và không bị thay đổi.


In [ ]:
_show_table('TBL02', 'public_test')
_show_table('TBL03', 'public_test')
_show_table('TBL04', 'public_test')

_show_figure('INV01', 'public_test', 1)
_show_figure('INV02', 'public_test', 1)
_show_figure('INV03', 'public_test', 1)
_show_figure('INV04', 'public_test', 1)


## 03. Khám phá schema và mối quan hệ

**Mục tiêu:** Xác định kiểu vật lý và ngữ nghĩa của từng cột, cùng bằng chứng khóa chính và quan hệ giữa các bảng.

**Phương pháp:** Suy luận kiểu dữ liệu trên toàn bộ dòng đã đọc, kết hợp bằng chứng khóa ứng viên và kiểm tra số khóa lệch.

**Cách đọc và hạn chế:** Quan hệ ứng viên chưa được phê duyệt không được dùng để tính chỉ số; phép nối có rủi ro nhân bản bị chặn theo thiết kế.


In [ ]:
_show_table('TBL05', 'public_test')
_show_table('TBL12', 'public_test')

_show_figure('GLOBAL02', 'public_test', 1)
_show_figure('QUAL07', 'public_test', 1)


## 04. Chất lượng dữ liệu

**Mục tiêu:** Đo mức thiếu dữ liệu, trùng lặp, vi phạm quy tắc và mức độ phát hiện theo từng nguồn, kèm mẫu số rõ ràng.

**Phương pháp:** Mọi tỷ lệ đều chia cho mẫu số của chính trường hoặc chính quy tắc được hồ sơ; thiếu dữ liệu có cấu trúc tách riêng khỏi khoảng trống thực.

**Cách đọc và hạn chế:** Cờ thiếu dữ liệu và cờ trùng lặp là tín hiệu thống kê, chưa được xác định là bản ghi lỗi nếu chưa có đối chiếu rulebook.


In [ ]:
_show_table('TBL06', 'public_test')
_show_table('TBL07', 'public_test')
_show_table('TBL08', 'public_test')
_show_table('TBL19', 'public_test')

_show_figure('QUAL01', 'public_test', 1)
_show_figure('QUAL02', 'public_test', 1)
_show_figure('QUAL04', 'public_test', 1)
_show_figure('QUAL05', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 05. Phân tích định lượng

**Mục tiêu:** Mô tả phân phối các đo lường định lượng trên toàn bộ dân số hợp lệ, kèm thống kê bền vững và cờ ngoại lệ theo IQR.

**Phương pháp:** Tính quantile, MAD, lệch và kurtosis trên toàn bộ quan sát hợp lệ; ngưỡng ngoại lệ IQR chỉ gắn cờ, không tự động loại bỏ.

**Cách đọc và hạn chế:** Kiểu vật lý không bảo đảm ngữ nghĩa đo lường; cột định danh dạng số được xử lý riêng và không đưa vào phân tích phân phối đo lường.


In [ ]:
_show_table('TBL09', 'public_test')

_show_figure('NUM01', 'public_test', 2)
_show_figure('NUM02', 'public_test', 1)
_show_figure('NUM06', 'public_test', 1)
_show_figure('NUM08', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 06. Phân tích phân loại

**Mục tiêu:** Khảo sát phân phối tần suất của các trường phân loại, giữ nguyên nhóm Others và ghi nhận ứng viên bí danh thay vì tự gộp.

**Phương pháp:** Đếm tần suất theo giá trị đã chuẩn hóa, sắp xếp theo tần suất giảm dần và ghi rõ phạm vi quan sát của mỗi trường.

**Cách đọc và hạn chế:** Cao độ lớp phụ thuộc quy tắc chuẩn hóa giá trị; các giá trị tương tự nhau về nghĩa chỉ được báo cáo là ứng viên, không tự ý hợp nhất.


In [ ]:
_show_table('TBL10', 'public_test')

_show_figure('CAT01', 'public_test', 2)
_show_figure('CAT02', 'public_test', 1)
_show_figure('CAT03', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 07. Tương quan và phân tích đa biến

**Mục tiêu:** Đo mức liên hệ giữa các đo lường và các trường phân loại trên cùng grain, chỉ dùng các phép nối đã được phê duyệt.

**Phương pháp:** Tính Pearson, Spearman, Kendall, Cramér V và bảng chéo trong phạm vi cùng một nguồn hoặc phép nối không nhân bản.

**Cách đọc và hạn chế:** Tương quan không phải quan hệ nhân quả; phép nối ứng viên chỉ được hiển thị để đánh giá rủi ro nhân bản, không dùng cho tính chỉ số.


In [ ]:
_show_table('TBL12', 'public_test')

_show_figure('REL01', 'public_test', 1)
_show_figure('REL03', 'public_test', 1)
_show_figure('REL06', 'public_test', 1)
_show_figure('REL07', 'public_test', 1)
_show_figure('REL08', 'public_test', 1)
_show_figure('REL09', 'public_test', 1)
_show_figure('REL10', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 08. Hồ sơ thời gian và lịch

**Mục tiêu:** Xác định ngữ nghĩa trường ngày, khoảng phủ thời gian, số ngày có hoạt động và các kỳ bị thiếu theo lịch của split.

**Phương pháp:** Đọc và chuẩn hóa giá trị ngày theo quy tắc từng split, đối chiếu với lịch và báo cáo khoảng phủ theo doanh nghiệp.

**Cách đọc và hạn chế:** Mỗi trường ngày giữ vai trò sự kiện riêng nên các khoảng phủ không thể so sánh trực tiếp; thiếu dữ liệu theo kỳ không được coi là giá trị bằng không.


In [ ]:
_show_table('TBL11', 'public_test')

_show_figure('TIME01', 'public_test', 1)
_show_figure('TIME02', 'public_test', 1)
_show_figure('TIME03', 'public_test', 1)
_show_figure('TIME07', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 09. Phân tích theo từng doanh nghiệp

**Mục tiêu:** Trình bày hồ sơ, phân loại ngành và các chỉ số kinh doanh đủ điều kiện của từng doanh nghiệp trong split.

**Phương pháp:** Chỉ số được tính trên grain đã xác định, tuân thủ quy tắc trạng thái của rulebook split và chỉ hiển thị khi doanh nghiệp đủ điều kiện.

**Cách đọc và hạn chế:** Doanh nghiệp thiếu điều kiện bị ghi trạng thái rõ ràng, không được thay bằng số liệu của doanh nghiệp khác để lấp chỗ trống.


In [ ]:
_show_table('TBL13', 'public_test')
_show_table('TBL14', 'public_test')

_show_figure('BUS01', 'public_test', 2)
_show_figure('BUS03', 'public_test', 1)
_show_figure('BUS05', 'public_test', 1)
_show_figure('QUAL03', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:5200]))


## 10. Hồ sơ câu hỏi người dùng Public Test

**Mục tiêu:** Khảo sát siêu dữ liệu câu hỏi trong tệp questions.jsonl của Public Test mà không truy cập đáp án.

**Phương pháp:** Đặc trưng câu hỏi suy ra bằng luật văn phong và cấu trúc ngữ nghĩa câu hỏi.

**Cách đọc và hạn chế:** Split Public Test không có đáp án tham chiếu theo quy định; nhãn ý định là quy tắc suy ra.


In [ ]:
_show_table('TBL15', 'public_test')

_show_figure('QNA01', 'public_test', 1)
_show_figure('QNA02', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:2600]))


## 11. Phân tích đặc trưng câu hỏi Public Test

**Mục tiêu:** Trình bày phân bổ đặc trưng, từ khóa thời gian và cấu trúc câu hỏi của Public Test.

**Phương pháp:** Thống kê từ khóa thời gian, độ dài ký tự và tỷ lệ câu hỏi trên tập Public Test.

**Cách đọc và hạn chế:** Tập trung hoàn toàn vào câu hỏi của 14 doanh nghiệp Public Test; không có so sánh chéo với train.


In [ ]:
_show_table('TBL15', 'public_test')

_show_figure('QNA01', 'public_test', 1)
_show_figure('QNA03', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:2600]))


## 12. Quy tắc nghiệp vụ theo Rulebook Public Test

**Mục tiêu:** Tổng hợp các quy tắc định nghĩa chỉ số và chuẩn nghiệp vụ trong rulebook.md của Public Test.

**Phương pháp:** Trích xuất quy tắc theo bảng và mục từ rulebook.md của Public Test.

**Cách đọc và hạn chế:** Quy tắc của Public Test là chuẩn duy nhất áp dụng cho toàn bộ phân tích nghiệp vụ.


In [ ]:
_show_table('TBL16')


## 13. So sánh doanh nghiệp trên toàn hệ thống

**Mục tiêu:** Đối chiếu sự hiện diện của vai trò trường ngữ nghĩa giữa các doanh nghiệp và khả năng dựng biểu đồ theo nhóm.

**Phương pháp:** Phát hiện vai trò trường theo tên và đối chiếu trạng thái đủ điều kiện của từng nhóm biểu đồ theo doanh nghiệp.

**Cách đọc và hạn chế:** Cùng một vai trò ngữ nghĩa không đồng nghĩa với khóa nối tương đương; đủ điều kiện về schema không bảo đảm biểu đồ dựng được.


In [ ]:
_show_table('TBL17', 'public_test')

_show_figure('GLOBAL02', 'public_test', 1)
_show_figure('GLOBAL03', 'public_test', 1)
_show_figure('GLOBAL04', 'public_test', 1)
_show_figure('GLOBAL05', None, 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:2600]))


## 14. Phân tích dữ liệu toàn cục

**Mục tiêu:** Đối chiếu mức độ bao phủ giữa các split và giữa các doanh nghiệp ở mức siêu dữ liệu.

**Phương pháp:** Tổng hợp số tệp, số doanh nghiệp, số worksheet, số đối tượng CSDL và số phát hiện chất lượng theo từng split.

**Cách đọc và hạn chế:** Số lượng đối tượng siêu dữ liệu không phải số giao dịch và không được cộng chéo giữa các loại đối tượng.


In [ ]:
_show_table('TBL18')

_show_figure('GLOBAL01', 'public_test', 1)
_show_figure('GLOBAL06', 'public_test', 1)


## 15. Chất lượng và mức độ sẵn sàng cho tác nhân

**Mục tiêu:** Trình bày ledger kiểm định theo phase, trạng thái phát hiện chất lượng và danh mục phát hiện có bằng chứng.

**Phương pháp:** Mỗi gate ghi rõ phase, trạng thái, lý do và artifact chứng minh; mỗi phát hiện gắn mức độ, tỷ lệ và bằng chứng nguồn.

**Cách đọc và hạn chế:** Phân biệt phát hiện có bằng chứng với lỗi đã xác nhận; trạng thái BLOCKED là kết quả hợp lệ và không được nâng lên PASS để đạt hình thức.


In [ ]:
_show_table('TBL19', 'public_test')
_show_table('TBL20')

_show_figure('QUAL06', 'public_test', 1)
_show_figure('QUAL08', None, 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:2600]))


## 16. Kết luận khoa học và bàn giao

**Mục tiêu:** Tổng hợp kết luận khoa học, hạn chế đã xác định và dòng dẫn tới từng artifact có hash để tái lập.

**Phương pháp:** Mỗi kết luận gắn mức tin cậy và đường dẫn bằng chứng; thẻ chuyện khoa học đính kèm theo từng nhóm nghiên cứu.

**Cách đọc và hạn chế:** Không có kết luận nào vượt quá phạm vi bằng chứng đã ghi; phần chưa đủ bằng chứng được nêu rõ thay vì bị lược bỏ.


In [ ]:
_show_figure('GLOBAL07', 'public_test', 1)

display(Markdown(run.story_cards_markdown(
    'public_test'
)[:15600]))
